In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("1. Đang tải siêu dữ liệu (Metadata)...")
# Đường dẫn mặc định của Kaggle sau khi Add Data ISIC 2024
train_df = pd.read_csv('/kaggle/input/competitions/isic-2024-challenge/train-metadata.csv')
print(f"Kích thước dữ liệu ban đầu: {train_df.shape}")

1. Đang tải siêu dữ liệu (Metadata)...
Kích thước dữ liệu ban đầu: (401059, 55)


In [3]:
print("2. Bắt đầu Tiền xử lý Metadata theo FusionNetX...")
# Xác định các cột phân loại và cột số dựa trên cấu trúc dữ liệu ISIC 2024
categorical_cols = ['sex', 'anatom_site_general'] 
numerical_cols = ['age_approx', 'clin_size_long_diam_mm']

2. Bắt đầu Tiền xử lý Metadata theo FusionNetX...


In [4]:
# Điền giá trị bị thiếu (Missing values)
# Dữ liệu số điền bằng trung vị (median)
num_imputer = SimpleImputer(strategy='median')
train_df[numerical_cols] = num_imputer.fit_transform(train_df[numerical_cols])

In [5]:
# Dữ liệu phân loại điền bằng giá trị xuất hiện nhiều nhất (mode)
cat_imputer = SimpleImputer(strategy='most_frequent')
train_df[categorical_cols] = cat_imputer.fit_transform(train_df[categorical_cols])

In [6]:
print("3. Trích xuất đặc trưng mới (Feature Engineering)...")
# Tạo đặc trưng mới: lesion size ratio và shape index 
if 'clin_size_long_diam_mm' in train_df.columns:
    train_df['lesion_size_ratio'] = train_df['clin_size_long_diam_mm'] / (train_df['clin_size_long_diam_mm'].mean())
    train_df['shape_index'] = train_df['clin_size_long_diam_mm'] ** 2 

print("4. Chuẩn hóa & Mã hóa (Normalization & One-Hot Encoding)...")
# One-Hot Encoding cho các biến phân loại
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)

3. Trích xuất đặc trưng mới (Feature Engineering)...
4. Chuẩn hóa & Mã hóa (Normalization & One-Hot Encoding)...


In [7]:
# Chuẩn hóa các đặc trưng số theo từng bệnh nhân để giảm sự khác biệt giữa các cá nhân
if 'patient_id' in train_df.columns:
    for col in numerical_cols:
        train_df[col] = train_df.groupby('patient_id')[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
        # Xử lý các giá trị NaN có thể sinh ra nếu bệnh nhân chỉ có 1 ảnh (std = 0)
        train_df[col].fillna(0, inplace=True)
else:
    scaler = StandardScaler()
    train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])

print("5. Thiết lập chia Fold (Stratified Group K-Fold)...")
# Nhóm dữ liệu theo ID của bệnh nhân để đảm bảo tính tổng quát và chống rò rỉ
sgkf = StratifiedGroupKFold(n_splits=5)
train_df['fold'] = -1

for f, (train_idx, val_idx) in enumerate(sgkf.split(train_df, y=train_df['target'], groups=train_df['patient_id'])):
    train_df.loc[val_idx, 'fold'] = f

print("\n--- HOÀN TẤT CELL 1 ---")
print("Thống kê số lượng mẫu trong từng Fold (kiểm tra độ cân bằng):")
print(train_df['fold'].value_counts())

5. Thiết lập chia Fold (Stratified Group K-Fold)...

--- HOÀN TẤT CELL 1 ---
Thống kê số lượng mẫu trong từng Fold (kiểm tra độ cân bằng):
fold
1    80216
0    80211
2    80211
3    80211
4    80210
Name: count, dtype: int64


In [8]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [9]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

print("BƯỚC 2: Cấu hình tiền xử lý hình ảnh (Đã tích hợp khiên chống lỗi thiếu ảnh)...")

image_size = 224
transforms_val = A.Compose([
    A.Resize(image_size, image_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class ISICDataset(Dataset):
    def __init__(self, df, file_hdf, transform=None):
        self.df = df
        self.file_hdf = file_hdf
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        isic_id = self.df['isic_id'].iloc[idx]
        
        # --- BỘ GIẢM XÓC (TRY-EXCEPT) ---
        try:
            img_bytes = self.file_hdf[isic_id][()]
            img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except KeyError:
            # Nếu Kaggle làm thiếu ảnh, tự động tạo một bức ảnh đen (ảnh rỗng) để mô hình không bị sập
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        # --------------------------------
            
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']

        target = self.df['target'].iloc[idx]
        return img, torch.tensor(target, dtype=torch.float32)

print("✅ Đã cập nhật thành công lớp ISICDataset chống lỗi KeyError!")

BƯỚC 2: Cấu hình tiền xử lý hình ảnh (Đã tích hợp khiên chống lỗi thiếu ảnh)...
✅ Đã cập nhật thành công lớp ISICDataset chống lỗi KeyError!


In [10]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm

In [11]:
print("BƯỚC 3: Cấu hình GeM Pooling và Trích xuất Đặc trưng Hình ảnh (OOF)...")

# Định nghĩa GeM Pooling theo chuẩn bài báo
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps
        
    def forward(self, x):
        return self.gem(x, p=self.p, eps=self.eps)
        
    def gem(self, x, p=3, eps=1e-6):
        return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)

BƯỚC 3: Cấu hình GeM Pooling và Trích xuất Đặc trưng Hình ảnh (OOF)...


In [12]:
# Cấu hình thiết bị (GPU nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device}")

# Mảng lưu kết quả OOF cho ConvNeXtV2
oof_preds_convnext = np.zeros(len(train_df))

print("Khởi tạo mô hình ConvNeXtV2-Nano...")
# Tải mô hình ConvNeXtV2 từ timm (sử dụng pre-trained)
# Lưu ý: Lần đầu chạy có thể tốn chút thời gian để tải mô hình
model_convnext = timm.create_model('convnextv2_nano', pretrained=True, num_classes=0)
model_convnext.global_pool = GeM()
classifier = nn.Linear(model_convnext.num_features, 1)
full_model = nn.Sequential(model_convnext, classifier).to(device)

print("✅ Đã khởi tạo thành công lớp GeM và mô hình ConvNeXtV2!")

Sử dụng thiết bị: cuda
Khởi tạo mô hình ConvNeXtV2-Nano...


model.safetensors:   0%|          | 0.00/62.5M [00:00<?, ?B/s]

✅ Đã khởi tạo thành công lớp GeM và mô hình ConvNeXtV2!


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np

print("BƯỚC 3: Cấu hình GeM Pooling và Mô hình Trích xuất Hình ảnh...")

# 1. Định nghĩa lớp GeM Pooling theo chuẩn của bài báo
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps
        
    def forward(self, x):
        return self.gem(x, p=self.p, eps=self.eps)
        
    def gem(self, x, p=3, eps=1e-6):
        return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)

# 2. Kiểm tra và sử dụng GPU của Kaggle (P100 / T4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Đang sử dụng thiết bị tính toán: {device}")

# 3. Tải mô hình ConvNeXtV2-Nano
print("⏳ Đang tải trọng số ConvNeXtV2-Nano từ kho lưu trữ...")
model_convnext = timm.create_model('convnextv2_nano', pretrained=True, num_classes=0)

# 4. Thay thế lớp Pooling mặc định của mô hình bằng GeM Pooling (Điểm "ăn tiền" của FusionNetX)
model_convnext.global_pool = GeM()

# 5. Gắn thêm một lớp Linear cuối cùng để xuất ra 1 con số (xác suất ung thư)
classifier = nn.Linear(model_convnext.num_features, 1)
full_model = nn.Sequential(model_convnext, classifier).to(device)

print("✅ Đã khởi tạo thành công lớp GeM và mô hình ConvNeXtV2!")

BƯỚC 3: Cấu hình GeM Pooling và Mô hình Trích xuất Hình ảnh...
🖥️ Đang sử dụng thiết bị tính toán: cuda
⏳ Đang tải trọng số ConvNeXtV2-Nano từ kho lưu trữ...
✅ Đã khởi tạo thành công lớp GeM và mô hình ConvNeXtV2!


In [14]:
import h5py
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import torch

print("BƯỚC 4: Trích xuất đặc trưng OOF (Out-of-Fold) bằng ConvNeXtV2...")

# Mở file HDF5 chứa ảnh từ bộ dữ liệu ISIC 2024
fp_hdf = h5py.File('/kaggle/input/competitions/isic-2024-challenge/test-image.hdf5', 'r')

# Khởi tạo mảng Numpy để lưu trữ xác suất dự đoán cho toàn bộ dữ liệu
oof_preds_convnext = np.zeros(len(train_df))

# Đưa mô hình về chế độ đánh giá (Inference mode), tắt Dropout và BatchNorm
full_model.eval()

print("🚀 Bắt đầu chạy trích xuất đặc trưng (Sử dụng GPU)...")

# Vòng lặp chạy qua 5 Folds đã chia ở Cell 1
for fold in range(5):
    print(f"\n--- Đang xử lý FOLD {fold} ---")
    
    # Lấy dữ liệu của tập Validation trong Fold hiện tại
    val_idx = train_df[train_df['fold'] == fold].index
    val_data = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Khởi tạo Dataset và DataLoader
    val_dataset = ISICDataset(val_data, fp_hdf, transform=transforms_val)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    
    val_preds = []
    
    # Tắt tính toán Gradient để tiết kiệm RAM và tăng tốc tối đa
    with torch.no_grad():
        for images, _ in tqdm(val_loader, desc=f"Dự đoán Fold {fold}"):
            images = images.to(device)
            outputs = full_model(images)
            
            # Sử dụng hàm Sigmoid để chuyển output thô thành xác suất [0, 1]
            probs = torch.sigmoid(outputs).squeeze(-1).cpu().numpy()
            val_preds.extend(probs)
            
    # Lưu kết quả dự đoán của Fold hiện tại vào đúng vị trí index gốc
    oof_preds_convnext[val_idx] = val_preds

# Đóng file HDF5 để giải phóng bộ nhớ
fp_hdf.close()

# Gắn cột đặc trưng hình ảnh vừa trích xuất vào bảng Metadata gốc
train_df['cnn_oof_convnext'] = oof_preds_convnext

print("\n🎉 Đã hoàn thành trích xuất đặc trưng ảnh và gắn vào Metadata!")
print("Xem thử 5 dòng đầu tiên của dữ liệu mới:")
print(train_df[['isic_id', 'target', 'cnn_oof_convnext']].head())

BƯỚC 4: Trích xuất đặc trưng OOF (Out-of-Fold) bằng ConvNeXtV2...
🚀 Bắt đầu chạy trích xuất đặc trưng (Sử dụng GPU)...

--- Đang xử lý FOLD 0 ---


Dự đoán Fold 0: 100%|██████████| 1254/1254 [05:34<00:00,  3.75it/s]



--- Đang xử lý FOLD 1 ---


Dự đoán Fold 1: 100%|██████████| 1254/1254 [05:30<00:00,  3.79it/s]



--- Đang xử lý FOLD 2 ---


...


--- Đang xử lý FOLD 3 ---


Dự đoán Fold 3: 100%|██████████| 1254/1254 [05:29<00:00,  3.81it/s]



--- Đang xử lý FOLD 4 ---


Dự đoán Fold 4: 100%|██████████| 1254/1254 [05:20<00:00,  3.91it/s]


🎉 Đã hoàn thành trích xuất đặc trưng ảnh và gắn vào Metadata!
Xem thử 5 dòng đầu tiên của dữ liệu mới:
        isic_id  target  cnn_oof_convnext
0  ISIC_0015670       0          0.519954
1  ISIC_0015845       0          0.519954
2  ISIC_0015864       0          0.519954
3  ISIC_0015902       0          0.519954
4  ISIC_0024200       0          0.519954


In [15]:
print("💾 Đang nén và lưu dữ liệu...")
train_df.to_parquet('/kaggle/working/fusionnetx_data.parquet', index=False)
print("✅ Đã lưu an toàn toàn bộ công sức vào file fusionnetx_data.parquet!")

💾 Đang nén và lưu dữ liệu...
✅ Đã lưu an toàn toàn bộ công sức vào file fusionnetx_data.parquet!


In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc
import gc
import warnings
warnings.filterwarnings('ignore')

print("1. Đọc dữ liệu từ file Parquet (RAM đang trống trải)...")
train_df = pd.read_parquet('/kaggle/working/fusionnetx_data.parquet')

print("2. TỊCH THU TOÀN BỘ 'PHAO THI' (Chống Rò rỉ dữ liệu)...")
# Danh sách các từ khóa cấm (chứa đáp án hoặc định danh)
leaky_keywords = ['iddx', 'mel_', 'benign', 'diagnosis', 'target', 'fold', 
                  'isic_id', 'patient_id', 'lesion_id', 'concomitant', 'attribution']

# Chỉ giữ lại những cột KHÔNG chứa từ khóa cấm
features = [c for c in train_df.columns if not any(k in c.lower() for k in leaky_keywords)]
print(f"✅ Số lượng đặc trưng hợp lệ (đã lọc phao thi): {len(features)} cột")

X = train_df[features].copy()
y = train_df['target'].copy()
folds = train_df['fold'].copy()

print("3. Đang dọn dẹp khoảng trống (NaN) và ép kiểu cho CatBoost...")
cat_cols = X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).replace('nan', 'Unknown')
    X[col] = X[col].astype('category')

# Xóa bảng gốc để giải phóng thêm RAM
del train_df
gc.collect()

print("4. Cấu hình mô hình (Theo đúng Bảng 2 bài báo FusionNetX)...")
lgb_params = {
    'objective': 'binary', 'boosting_type': 'gbdt', 'random_state': 42,
    'learning_rate': 0.03231, 'max_depth': 4, 'num_leaves': 103,
    'colsample_bytree': 0.83296, 'min_child_samples': 85,
    'scale_pos_weight': 2.7984, 'n_estimators': 200, 'device': 'cpu', 
    'n_jobs': 2, 'verbose': -1
}

cat_params = {
    'loss_function': 'Logloss', 'random_state': 42, 'learning_rate': 0.06936,
    'depth': 7, 'subsample': 0.6249, 'min_data_in_leaf': 24,
    'scale_pos_weight': 2.6149, 'iterations': 250, 'task_type': 'CPU', 
    'thread_count': 2, 'verbose': 0
}

xgb_params = {
    'objective': 'binary:logistic', 'tree_method': 'hist', 'random_state': 42,
    'learning_rate': 0.08501, 'max_depth': 6, 'subsample': 0.6013,
    'colsample_bytree': 0.84378, 'min_child_weight': 85,
    'scale_pos_weight': 3.2944, 'n_estimators': 100, 
    'enable_categorical': True, 'device': 'cpu', 'n_jobs': 2
}

oof_lgb = np.zeros(len(y))
oof_cat = np.zeros(len(y))
oof_xgb = np.zeros(len(y))
cat_features_indices = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

print("🚀 BẮT ĐẦU HUẤN LUYỆN VOTING ENSEMBLE (THI THẬT)...")
for fold in range(5):
    print(f"\n--- Đang huấn luyện FOLD {fold} ---")
    trn_idx, val_idx = (folds != fold), (folds == fold)
    
    print("  + Chạy LightGBM...")
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X[trn_idx], y[trn_idx])
    oof_lgb[val_idx] = model_lgb.predict_proba(X[val_idx])[:, 1]
    
    print("  + Chạy CatBoost...")
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(X[trn_idx], y[trn_idx], cat_features=cat_features_indices)
    oof_cat[val_idx] = model_cat.predict_proba(X[val_idx])[:, 1]
    
    print("  + Chạy XGBoost...")
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X[trn_idx], y[trn_idx])
    oof_xgb[val_idx] = model_xgb.predict_proba(X[val_idx])[:, 1]
    
    del model_lgb, model_cat, model_xgb, trn_idx, val_idx
    gc.collect()

print("\n🎉 HOÀN TẤT TỔNG HỢP (SOFT VOTING)...")
oof_preds = (oof_lgb + oof_cat + oof_xgb) / 3.0

print("📏 ĐANG TÍNH TOÁN ĐIỂM SỐ CHUẨN...")
final_auc = roc_auc_score(y, oof_preds)

# Tính pAUC (TPR >= 80%)
fpr, tpr, thresholds = roc_curve(y, oof_preds)
min_tpr = 0.8
valid_idx = tpr >= min_tpr

if any(valid_idx):
    tpr_valid = tpr[valid_idx]
    fpr_valid = fpr[valid_idx]
    pauc_score = auc(fpr_valid, tpr_valid)
    print(f"🌟 ĐIỂM FULL AUC: {final_auc:.5f}")
    print(f"🎯 ĐIỂM pAUC THỰC TẾ CỦA BẠN: {pauc_score:.5f}")
else:
    print(f"🌟 ĐIỂM FULL AUC: {final_auc:.5f}")
    print("Mô hình chưa đạt đủ 80% TPR để tính pAUC.")

1. Đọc dữ liệu từ file Parquet (RAM đang trống trải)...
2. TỊCH THU TOÀN BỘ 'PHAO THI' (Chống Rò rỉ dữ liệu)...
✅ Số lượng đặc trưng hợp lệ (đã lọc phao thi): 48 cột
3. Đang dọn dẹp khoảng trống (NaN) và ép kiểu cho CatBoost...
4. Cấu hình mô hình (Theo đúng Bảng 2 bài báo FusionNetX)...
🚀 BẮT ĐẦU HUẤN LUYỆN VOTING ENSEMBLE (THI THẬT)...

--- Đang huấn luyện FOLD 0 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 1 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 2 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 3 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 4 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

🎉 HOÀN TẤT TỔNG HỢP (SOFT VOTING)...
📏 ĐANG TÍNH TOÁN ĐIỂM SỐ CHUẨN...
🌟 ĐIỂM FULL AUC: 0.95489
🎯 ĐIỂM pAUC THỰC TẾ CỦA BẠN: 0.92130


In [17]:
from sklearn.metrics import roc_auc_score
import numpy as np

def isic_2024_pauc(solution, submission, min_tpr=0.80):
    """
    Hàm tính điểm pAUC chính thức của ban tổ chức cuộc thi ISIC 2024
    """
    # Đảo ngược nhãn (0 thành 1, 1 thành 0) theo thuật toán của Kaggle
    v_gt = abs(np.asarray(solution) - 1)
    v_pred = np.array([1.0 - x for x in submission])
    max_fpr = abs(1 - min_tpr)
    
    # Tính pAUC bị scale của sklearn
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    
    # Giải mã (De-scale) về lại diện tích thực tế (tối đa là 0.20000)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return partial_auc

# Chấm điểm mô hình của bạn
diem_chuan = isic_2024_pauc(y, oof_preds)
print(f"🏆 ĐIỂM pAUC CHÍNH THỨC CỦA BẠN (CHUẨN KAGGLE): {diem_chuan:.5f}")

🏆 ĐIỂM pAUC CHÍNH THỨC CỦA BẠN (CHUẨN KAGGLE): 0.16188


In [20]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import timm
from tqdm import tqdm
import h5py
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import gc
import warnings
warnings.filterwarnings('ignore')

print("1. Đang nạp lại dữ liệu từ file Parquet và định nghĩa Dataset...")
# Đọc lại file đã lưu (file này đã có sẵn cột OOF của ConvNeXtV2)
train_df = pd.read_parquet('/kaggle/working/fusionnetx_data.parquet')

# Khởi tạo lại các hàm biến đổi ảnh và Dataset
image_size = 224
transforms_val = A.Compose([
    A.Resize(image_size, image_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class ISICDataset(Dataset):
    def __init__(self, df, file_hdf, transform=None):
        self.df = df
        self.file_hdf = file_hdf
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        isic_id = self.df['isic_id'].iloc[idx]
        try:
            img_bytes = self.file_hdf[isic_id][()]
            img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except KeyError:
            # Khiên chống lỗi thiếu ảnh
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']

        target = self.df['target'].iloc[idx]
        return img, torch.tensor(target, dtype=torch.float32)

print("2. BƯỚC 3.2: Khởi tạo mô hình ViT-Tiny và ViT-Small...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

oof_preds_vit_tiny = np.zeros(len(train_df))
oof_preds_vit_small = np.zeros(len(train_df))

print("⏳ Đang tải trọng số ViT-Tiny...")
model_vit_tiny = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0)
classifier_tiny = nn.Linear(model_vit_tiny.num_features, 1)
full_model_tiny = nn.Sequential(model_vit_tiny, classifier_tiny).to(device)
full_model_tiny.eval()

print("⏳ Đang tải trọng số ViT-Small...")
model_vit_small = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
classifier_small = nn.Linear(model_vit_small.num_features, 1)
full_model_small = nn.Sequential(model_vit_small, classifier_small).to(device)
full_model_small.eval()

print("🚀 Bắt đầu trích xuất đặc trưng OOF cho ViT-Tiny và ViT-Small...")
# Trỏ đúng đường dẫn file ảnh
fp_hdf = h5py.File('/kaggle/input/competitions/isic-2024-challenge/train-image.hdf5', 'r')

for fold in range(5):
    print(f"\n--- Đang xử lý FOLD {fold} ---")
    val_idx = train_df[train_df['fold'] == fold].index
    val_data = train_df.iloc[val_idx].reset_index(drop=True)
    
    val_dataset = ISICDataset(val_data, fp_hdf, transform=transforms_val)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    
    val_preds_tiny = []
    val_preds_small = []
    
    with torch.no_grad():
        for images, _ in tqdm(val_loader, desc=f"Dự đoán ViT Fold {fold}"):
            images = images.to(device)
            
            # Dự đoán bằng ViT-Tiny
            out_tiny = full_model_tiny(images)
            probs_tiny = torch.sigmoid(out_tiny).squeeze(-1).cpu().numpy()
            val_preds_tiny.extend(probs_tiny)
            
            # Dự đoán bằng ViT-Small
            out_small = full_model_small(images)
            probs_small = torch.sigmoid(out_small).squeeze(-1).cpu().numpy()
            val_preds_small.extend(probs_small)
            
    oof_preds_vit_tiny[val_idx] = val_preds_tiny
    oof_preds_vit_small[val_idx] = val_preds_small

fp_hdf.close()

# Gắn 2 cột đặc trưng mới này vào bảng dữ liệu
train_df['cnn_oof_vit_tiny'] = oof_preds_vit_tiny
train_df['cnn_oof_vit_small'] = oof_preds_vit_small

# Dọn dẹp GPU ngay lập tức
del full_model_tiny, full_model_small, model_vit_tiny, model_vit_small
torch.cuda.empty_cache()
gc.collect()

print("\n💾 Đang ghi đè dữ liệu mới vào ổ cứng...")
train_df.to_parquet('/kaggle/working/fusionnetx_data.parquet', index=False)

print("🎉 HOÀN THÀNH 100% CÔNG LỰC TRÍCH XUẤT ẢNH!")
print("Dữ liệu của bạn hiện tại đã có đủ 3 cột OOF: ConvNeXtV2, ViT-Tiny, ViT-Small và đã được lưu an toàn.")

1. Đang nạp lại dữ liệu từ file Parquet và định nghĩa Dataset...
2. BƯỚC 3.2: Khởi tạo mô hình ViT-Tiny và ViT-Small...
⏳ Đang tải trọng số ViT-Tiny...
⏳ Đang tải trọng số ViT-Small...
🚀 Bắt đầu trích xuất đặc trưng OOF cho ViT-Tiny và ViT-Small...

--- Đang xử lý FOLD 0 ---


Dự đoán ViT Fold 0: 100%|██████████| 1254/1254 [08:12<00:00,  2.55it/s]



--- Đang xử lý FOLD 1 ---


Dự đoán ViT Fold 1: 100%|██████████| 1254/1254 [07:48<00:00,  2.68it/s]



--- Đang xử lý FOLD 2 ---


...


--- Đang xử lý FOLD 3 ---


Dự đoán ViT Fold 3: 100%|██████████| 1254/1254 [07:44<00:00,  2.70it/s]



--- Đang xử lý FOLD 4 ---


Dự đoán ViT Fold 4: 100%|██████████| 1254/1254 [07:43<00:00,  2.70it/s]



💾 Đang ghi đè dữ liệu mới vào ổ cứng...
🎉 HOÀN THÀNH 100% CÔNG LỰC TRÍCH XUẤT ẢNH!
Dữ liệu của bạn hiện tại đã có đủ 3 cột OOF: ConvNeXtV2, ViT-Tiny, ViT-Small và đã được lưu an toàn.


In [21]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("1. Đọc dữ liệu 'Full Giáp' từ file Parquet...")
train_df = pd.read_parquet('/kaggle/working/fusionnetx_data.parquet')

print("2. TỊCH THU TOÀN BỘ 'PHAO THI' (Chống Rò rỉ dữ liệu)...")
leaky_keywords = ['iddx', 'mel_', 'benign', 'diagnosis', 'target', 'fold', 
                  'isic_id', 'patient_id', 'lesion_id', 'concomitant', 'attribution']

# Chỉ giữ lại những cột KHÔNG chứa từ khóa cấm
features = [c for c in train_df.columns if not any(k in c.lower() for k in leaky_keywords)]
print(f"✅ Số lượng đặc trưng hợp lệ để đưa vào Ensemble: {len(features)} cột")
# (Trong số này chắc chắn đã bao gồm cnn_oof_convnext, cnn_oof_vit_tiny, cnn_oof_vit_small)

X = train_df[features].copy()
y = train_df['target'].copy()
folds = train_df['fold'].copy()

print("3. Đang dọn dẹp và ép kiểu cho CatBoost...")
cat_cols = X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).replace('nan', 'Unknown')
    X[col] = X[col].astype('category')

del train_df
gc.collect()

print("4. Cấu hình mô hình Tree-based (Bảng 2 - FusionNetX)...")
lgb_params = {
    'objective': 'binary', 'boosting_type': 'gbdt', 'random_state': 42,
    'learning_rate': 0.03231, 'max_depth': 4, 'num_leaves': 103,
    'colsample_bytree': 0.83296, 'min_child_samples': 85,
    'scale_pos_weight': 2.7984, 'n_estimators': 200, 'device': 'cpu', 
    'n_jobs': 2, 'verbose': -1
}

cat_params = {
    'loss_function': 'Logloss', 'random_state': 42, 'learning_rate': 0.06936,
    'depth': 7, 'subsample': 0.6249, 'min_data_in_leaf': 24,
    'scale_pos_weight': 2.6149, 'iterations': 250, 'task_type': 'CPU', 
    'thread_count': 2, 'verbose': 0
}

xgb_params = {
    'objective': 'binary:logistic', 'tree_method': 'hist', 'random_state': 42,
    'learning_rate': 0.08501, 'max_depth': 6, 'subsample': 0.6013,
    'colsample_bytree': 0.84378, 'min_child_weight': 85,
    'scale_pos_weight': 3.2944, 'n_estimators': 100, 
    'enable_categorical': True, 'device': 'cpu', 'n_jobs': 2
}

oof_lgb = np.zeros(len(y))
oof_cat = np.zeros(len(y))
oof_xgb = np.zeros(len(y))
cat_features_indices = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

print("🚀 BẮT ĐẦU HUẤN LUYỆN VOTING ENSEMBLE (100% CÔNG LỰC)...")
for fold in range(5):
    print(f"\n--- Đang huấn luyện FOLD {fold} ---")
    trn_idx, val_idx = (folds != fold), (folds == fold)
    
    print("  + Chạy LightGBM...")
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X[trn_idx], y[trn_idx])
    oof_lgb[val_idx] = model_lgb.predict_proba(X[val_idx])[:, 1]
    
    print("  + Chạy CatBoost...")
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(X[trn_idx], y[trn_idx], cat_features=cat_features_indices)
    oof_cat[val_idx] = model_cat.predict_proba(X[val_idx])[:, 1]
    
    print("  + Chạy XGBoost...")
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X[trn_idx], y[trn_idx])
    oof_xgb[val_idx] = model_xgb.predict_proba(X[val_idx])[:, 1]
    
    del model_lgb, model_cat, model_xgb, trn_idx, val_idx
    gc.collect()

print("\n🎉 HOÀN TẤT TỔNG HỢP (SOFT VOTING)...")
oof_preds = (oof_lgb + oof_cat + oof_xgb) / 3.0

print("📏 ĐANG TÍNH TOÁN ĐIỂM SỐ CHUẨN KAGGLE...")
def isic_2024_pauc(solution, submission, min_tpr=0.80):
    v_gt = abs(np.asarray(solution) - 1)
    v_pred = np.array([1.0 - x for x in submission])
    max_fpr = abs(1 - min_tpr)
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return partial_auc

diem_chuan = isic_2024_pauc(y, oof_preds)
print(f"🏆 ĐIỂM pAUC CHÍNH THỨC CỦA BẠN: {diem_chuan:.5f}")

1. Đọc dữ liệu 'Full Giáp' từ file Parquet...
2. TỊCH THU TOÀN BỘ 'PHAO THI' (Chống Rò rỉ dữ liệu)...
✅ Số lượng đặc trưng hợp lệ để đưa vào Ensemble: 50 cột
3. Đang dọn dẹp và ép kiểu cho CatBoost...
4. Cấu hình mô hình Tree-based (Bảng 2 - FusionNetX)...
🚀 BẮT ĐẦU HUẤN LUYỆN VOTING ENSEMBLE (100% CÔNG LỰC)...

--- Đang huấn luyện FOLD 0 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 1 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 2 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 3 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

--- Đang huấn luyện FOLD 4 ---
  + Chạy LightGBM...
  + Chạy CatBoost...
  + Chạy XGBoost...

🎉 HOÀN TẤT TỔNG HỢP (SOFT VOTING)...
📏 ĐANG TÍNH TOÁN ĐIỂM SỐ CHUẨN KAGGLE...
🏆 ĐIỂM pAUC CHÍNH THỨC CỦA BẠN: 0.16106
